# Nuvii Nail AI — Low-Memory LoRA Training

This notebook replaces the FLUX.2 training attempt that exceeded Kaggle's system RAM.

It trains a nail-domain LoRA on **Segmind SSD-1B**, a smaller SDXL-family model, using the existing full Nuvii dataset.

## Before running

1. Attach `nuvii_training_dataset_v1_full.zip` using **Add Input**.
2. Set **Accelerator → GPU T4 x2**.
3. Set **Internet → On**.
4. Run every cell in order.

The notebook uses only GPU 0 and leaves GPU 1 unused. This is intentional because two 15 GB GPUs do not combine into one 30 GB memory pool for this training script.

> The repository version includes the Accelerate logger fix: `--report_to=tensorboard`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile, json, time, re

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
TRAIN_DIR = WORK_ROOT / 'nuvii_ssd1b_train'
OUTPUT_DIR = WORK_ROOT / 'nuvii_ssd1b_lora'
CACHE_DIR = WORK_ROOT / 'hf_cache'
SCRIPT_DIR = WORK_ROOT / 'diffusers_source'
LOG_PATH = WORK_ROOT / 'nuvii_ssd1b_training.log'

print('GPU information:')
subprocess.run([
    'nvidia-smi', '--query-gpu=index,name,memory.total',
    '--format=csv,noheader'
], check=False)

print('\nSystem memory:')
subprocess.run(['free', '-h'], check=False)

print('\nWorking disk:')
subprocess.run(['df', '-h', '/kaggle/working'], check=False)


In [ ]:
# Find the full 196 MB Nuvii dataset and extract only the training split.

if TRAIN_DIR.exists():
    shutil.rmtree(TRAIN_DIR)
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

direct_train_folders = [
    p for p in INPUT_ROOT.rglob('train_imagefolder') if p.is_dir()
]

if direct_train_folders:
    source_train = direct_train_folders[0]
    print('Using Kaggle-unpacked training folder:', source_train)
    for item in source_train.iterdir():
        if item.is_file():
            shutil.copy2(item, TRAIN_DIR / item.name)
else:
    exact = list(INPUT_ROOT.rglob('nuvii_training_dataset_v1_full.zip'))
    fallback = [
        p for p in INPUT_ROOT.rglob('*.zip')
        if 'nuvii' in p.name.lower() and 'full' in p.name.lower()
    ]
    zip_matches = exact or fallback

    if not zip_matches:
        visible = '\n'.join(str(p) for p in list(INPUT_ROOT.rglob('*'))[:100])
        raise FileNotFoundError(
            'The full Nuvii dataset is not attached to this notebook. '
            'Use Add Input and attach nuvii_training_dataset_v1_full.zip.\n\n'
            f'Visible files:\n{visible}'
        )

    archive_path = zip_matches[0]
    print('Found full dataset ZIP:', archive_path)

    with zipfile.ZipFile(archive_path, 'r') as zf:
        members = [
            name for name in zf.namelist()
            if '/train_imagefolder/' in name and not name.endswith('/')
        ]
        if not members:
            raise FileNotFoundError('train_imagefolder was not found inside the ZIP.')

        for member in members:
            target = TRAIN_DIR / Path(member).name
            with zf.open(member) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

image_exts = {'.png', '.jpg', '.jpeg', '.webp'}
images = [p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in image_exts]
captions = list(TRAIN_DIR.glob('*.txt'))
metadata = TRAIN_DIR / 'metadata.jsonl'

print('\nTraining directory:', TRAIN_DIR)
print('Images:', len(images))
print('Caption files:', len(captions))
print('metadata.jsonl exists:', metadata.exists())

if len(images) != 140:
    raise RuntimeError(f'Expected 140 training images, found {len(images)}.')
if len(captions) != 140:
    raise RuntimeError(f'Expected 140 caption files, found {len(captions)}.')
if not metadata.exists():
    raise FileNotFoundError('metadata.jsonl is missing from the training folder.')

print('\nDATASET READY')


In [ ]:
# Install the current official Hugging Face Diffusers training stack.
# This can take several minutes. Warnings are okay unless a red traceback appears.

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/huggingface/diffusers.git',
    'transformers', 'accelerate', 'datasets', 'peft',
    'bitsandbytes', 'xformers', 'safetensors', 'sentencepiece'
], check=True)

print('Training libraries installed.')

# Kaggle may preinstall an old torchao release that current PEFT rejects.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
print('Removed incompatible preinstalled torchao if present.')


In [ ]:
# Download the official SDXL text-to-image LoRA training script.
import urllib.request

if SCRIPT_DIR.exists():
    shutil.rmtree(SCRIPT_DIR)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

SCRIPT_PATH = SCRIPT_DIR / 'train_text_to_image_lora_sdxl.py'
SCRIPT_URL = (
    'https://raw.githubusercontent.com/huggingface/diffusers/main/'
    'examples/text_to_image/train_text_to_image_lora_sdxl.py'
)

urllib.request.urlretrieve(SCRIPT_URL, SCRIPT_PATH)

if not SCRIPT_PATH.exists() or SCRIPT_PATH.stat().st_size < 10000:
    raise RuntimeError('The official training script did not download correctly.')

print('Training script:', SCRIPT_PATH)
print('Size:', round(SCRIPT_PATH.stat().st_size / 1024, 1), 'KB')


## Train the Nuvii LoRA

This cell remains active until training finishes. It prints a small heartbeat instead of flooding the notebook output.

Expected runtime varies with Kaggle load. Checkpoints are saved every 200 steps.


In [ ]:
# Start low-memory SSD-1B LoRA training.

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if LOG_PATH.exists():
    LOG_PATH.unlink()

command = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--num_processes=1',
    '--mixed_precision=fp16',
    str(SCRIPT_PATH),
    '--pretrained_model_name_or_path=segmind/SSD-1B',
    '--variant=fp16',
    f'--train_data_dir={TRAIN_DIR}',
    f'--output_dir={OUTPUT_DIR}',
    f'--cache_dir={CACHE_DIR}',
    '--resolution=512',
    '--center_crop',
    '--train_batch_size=1',
    '--gradient_accumulation_steps=4',
    '--gradient_checkpointing',
    '--max_train_steps=1200',
    '--learning_rate=1e-4',
    '--lr_scheduler=cosine',
    '--lr_warmup_steps=100',
    '--snr_gamma=5.0',
    '--checkpointing_steps=200',
    '--checkpoints_total_limit=3',
    '--mixed_precision=fp16',
    '--enable_xformers_memory_efficient_attention',
    '--use_8bit_adam',
    '--allow_tf32',
    '--dataloader_num_workers=0',
    '--rank=8',
    '--seed=42',
    '--report_to=tensorboard',
]

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['PYTHONUNBUFFERED'] = '1'
env['HF_HOME'] = str(CACHE_DIR)
env['TRANSFORMERS_CACHE'] = str(CACHE_DIR / 'transformers')
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Launching training...')
print('Model: segmind/SSD-1B')
print('Steps: 1200')
print('Resolution: 512')
print('Log:', LOG_PATH)

with open(LOG_PATH, 'w', buffering=1) as log_handle:
    process = subprocess.Popen(
        command,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env,
        cwd=str(WORK_ROOT),
    )

    last_size = -1
    while process.poll() is None:
        time.sleep(60)
        size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0
        text = LOG_PATH.read_text(errors='ignore') if LOG_PATH.exists() else ''
        step_matches = re.findall(r'(?:Steps|step|Step)[: ]+([0-9]+)', text[-20000:])
        last_step = step_matches[-1] if step_matches else 'loading'

        gpu = subprocess.run([
            'nvidia-smi',
            '--query-gpu=index,memory.used,utilization.gpu',
            '--format=csv,noheader'
        ], capture_output=True, text=True, check=False)

        print(f'Heartbeat — status: {last_step}; log: {round(size/1024/1024, 2)} MB')
        print(gpu.stdout.strip())

exit_code = process.returncode
log_text = LOG_PATH.read_text(errors='ignore') if LOG_PATH.exists() else ''

print('\nTraining process ended with exit code:', exit_code)

if exit_code != 0:
    print('\nLAST 100 LOG LINES:\n')
    print('\n'.join(log_text.splitlines()[-100:]))
    raise RuntimeError('Training stopped with an error. The final log lines are shown above.')

weights = list(OUTPUT_DIR.rglob('*.safetensors'))
print('\nTRAINING FINISHED')
print('LoRA weight files:')
for weight in weights:
    print(weight, '-', round(weight.stat().st_size / (1024**2), 2), 'MB')

if not weights:
    raise FileNotFoundError('Training ended but no .safetensors LoRA file was found.')


In [ ]:
# Generate a few test nail designs with the trained LoRA.
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image

SAMPLE_DIR = WORK_ROOT / 'nuvii_ssd1b_samples'
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

pipe = StableDiffusionXLPipeline.from_pretrained(
    'segmind/SSD-1B',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe.load_lora_weights(str(OUTPUT_DIR))
pipe.enable_xformers_memory_efficient_attention()
pipe.to('cuda')

prompts = [
    'NUV11NAIL, single isolated long almond press-on nail, front view, centered on plain white background, milky nude glossy base, silver isolated chrome bow near the cuticle, four-point stars, subtle glitter fade',
    'NUV11NAIL, single isolated medium almond press-on nail, front view, centered on plain white background, translucent sage jelly base, pearlescent raised shell ripples, clear gel bubbles, glossy finish',
    'NUV11NAIL, single isolated long stiletto press-on nail, front view, centered on plain white background, cobalt blue base, gold isolated chrome organic frame, tiny caviar beads, futuristic luxury nail art',
    'NUV11NAIL, single isolated medium square press-on nail, front view, centered on plain white background, blush nude base, rounded burgundy French tip, thin gold chrome smile line, elegant glossy finish',
]

negative = 'hand, finger, skin, multiple nails, nail set, text, watermark, package, bottle, salon scene, cropped nail, blurry, deformed'

for index, prompt in enumerate(prompts, start=1):
    generator = torch.Generator(device='cuda').manual_seed(1000 + index)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        width=512,
        height=768,
        num_inference_steps=30,
        guidance_scale=8.0,
        generator=generator,
    ).images[0]
    path = SAMPLE_DIR / f'nuvii_sample_{index}.png'
    image.save(path)
    print('Saved:', path)

del pipe
torch.cuda.empty_cache()


In [ ]:
# Package the LoRA, checkpoints, samples, training log, and metadata for download.

PACKAGE_DIR = WORK_ROOT / 'nuvii_ssd1b_lora_package'
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(OUTPUT_DIR, PACKAGE_DIR / 'lora_output')

if SAMPLE_DIR.exists():
    shutil.copytree(SAMPLE_DIR, PACKAGE_DIR / 'samples')

if LOG_PATH.exists():
    shutil.copy2(LOG_PATH, PACKAGE_DIR / LOG_PATH.name)

shutil.copy2(TRAIN_DIR / 'metadata.jsonl', PACKAGE_DIR / 'training_metadata.jsonl')

readme = '''# Nuvii SSD-1B Nail LoRA\n\nBase model: segmind/SSD-1B\nTrigger word: NUV11NAIL\nTraining images: 140\nResolution: 512\nTraining steps: 1200\nLoRA rank: 8\n'''
(PACKAGE_DIR / 'README.md').write_text(readme)

zip_path = shutil.make_archive(
    str(WORK_ROOT / 'nuvii_ssd1b_lora_outputs'),
    'zip',
    root_dir=PACKAGE_DIR,
)

print('DOWNLOAD THIS FILE:')
print(zip_path)
print('Size:', round(Path(zip_path).stat().st_size / (1024**2), 2), 'MB')
